# 02 · Model loading and first coding inference
Downloads `Qwen/Qwen2.5-Coder-7B-Instruct` (~15 GB, once), loads it in bf16 with the standard full KV cache, and runs
one coding task. **Shut this notebook's kernel down before running notebook 03**, otherwise two copies of the
model compete for the 24 GB GPU.

In [ ]:
# Run from anywhere: find the project root and make it importable
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
PY = sys.executable
print("project root:", ROOT)

In [ ]:
# Hugging Face token: kept in this kernel's environment only, never written to disk or printed.
# Qwen2.5-Coder is not gated, so the token is optional, but it avoids anonymous rate limits.
import os, getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (Enter to skip): ").strip()
    if not os.environ["HF_TOKEN"]:
        del os.environ["HF_TOKEN"]
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
# Put the model cache on the persistent volume of the space
os.environ.setdefault("HF_HOME", str(Path.home() / "hf_cache"))
print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
!{PY} scripts/download_model.py --model Qwen/Qwen2.5-Coder-7B-Instruct

In [ ]:
from utils.config import load_config
from models.model_loader import load_model
from evaluation.memory_metrics import snapshot, gib

cfg = load_config()                      # configs/model_config.yaml + configs/experiment_config.yaml
model, tok, info = load_model(cfg)
info

In [ ]:
snap = snapshot()
print("model weights on GPU:", gib(info["gpu_model_memory_bytes"]), "GiB")
print("free after load     :", gib(snap["free_bytes"]), "of", gib(snap["total_bytes"]), "GiB")
print("advertised max context (max_position_embeddings):", info["max_position_embeddings"])
print("KV heads / layers / head_dim:", info["num_key_value_heads"], info["num_hidden_layers"], info["head_dim"])

### Chat template check

In [ ]:
from inference.generation import build_messages, encode_messages
from utils.dataset import load_samples
samples, _ = load_samples("datasets/sample_tasks.json")
s = samples[0]
ids = encode_messages(tok, build_messages(s))
print(len(ids), "prompt tokens\n")
print(tok.decode(ids))

### First full-cache coding inference

In [ ]:
from inference.generation import prepare_prompt
from inference.inference_runner import InferenceRunner
from evaluation.correctness import evaluate_sample

runner = InferenceRunner(model, tok, cfg)
ids, pinfo = prepare_prompt(tok, s, cfg)
r = runner.run(ids, pinfo["max_new_tokens"])
print(r["response"])

In [ ]:
ev = evaluate_sample(s, r["response"])
keys = ["output_tokens", "prefill_time_seconds", "ttft_seconds", "tpot_ms", "total_inference_seconds",
        "kv_cache_bytes", "kv_cache_bytes_theoretical_full", "peak_allocated_gpu_bytes", "peak_reserved_gpu_bytes"]
{**pinfo, **{k: r[k] for k in keys}, **{k: ev[k] for k in ("syntax_valid", "functional_correctness", "unit_test_pass_rate")}}

In [ ]:
import json
from utils.paths import results_dirs
out = results_dirs()["json"] / "first_inference.json"
out.write_text(json.dumps({"sample_id": s["sample_id"], "model_info": info, **pinfo, **r, **ev}, indent=2, default=str))
print("saved", out)

### Model and tokenizer tests (real model)

In [ ]:
# Frees this notebook's copy first so the test process can load its own
del runner, model; import gc, torch; gc.collect(); torch.cuda.empty_cache()
!KVR_TEST_MODEL=Qwen/Qwen2.5-Coder-7B-Instruct {PY} -m pytest tests/test_model.py tests/test_inference.py -q

**Now: Kernel → Shut Down Kernel** before opening notebook 03.